# Resume NER Training - FROM SCRATCH (No Pre-trained Models upadted)

This notebook trains a BiLSTM-CRF model completely from scratch.

**Key Difference from BERT approach:**
- ❌ No pre-trained models (BERT, etc.)
- ✅ Random weight initialization
- ✅ Builds vocabulary from your data
- ✅ Trains everything from scratch

## Setup Steps:
1. Enable GPU: Runtime → Change runtime type → GPU
2. Upload your dataset to Google Drive:
   - **RECOMMENDED**: `dataturks_resumes_fixed.json` (200 examples, excellent quality)
   - **Alternative**: `train_aggressive_cleaned.json` (5,943 examples, larger dataset)
3. Mount Google Drive and update the dataset path in Step 2
4. Run all cells below

## Optimizations Applied:
- ✅ Using high-quality dataset (DataTurks - email issues fixed)
- ✅ **Larger model**: 128 embedding dim, 384 hidden dim, 3 LSTM layers
- ✅ **Class weighting** (handles imbalanced data better)
- ✅ **Learning rate warmup** (3 epochs) + scheduling (reduces LR on plateau)
- ✅ **Weight decay** (L2 regularization) to prevent overfitting
- ✅ **Gradient clipping** to prevent exploding gradients
- ✅ More epochs (50) with early stopping (patience=10)
- ✅ Optimized learning rate (0.001) with warmup
- ✅ Balanced dropout (0.5) for better learning
- ✅ Fixed loss calculation (proper attention masks)

## Step 1: Install Dependencies

In [1]:
!pip install torch numpy tqdm

## Step 2: Load Dataset from Google Drive

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# Update the path below to match where your dataset is stored in Drive
# OPTION 1: DataTurks dataset (RECOMMENDED - better quality, 200 examples)
DRIVE_DATASET_PATH = '/content/drive/MyDrive/DATASETS/dataturks_resumes_fixed.json'  # ← HIGH QUALITY (200 examples)

# OPTION 2: Large dataset (5,960 examples - BUT LABELS ARE WRONG!)
# DRIVE_DATASET_PATH = '/content/drive/MyDrive/DATASETS/dataset-5000/train_cleaned.json'

# OPTION 3: Aggressive cleaned dataset (5,943 examples)
# DRIVE_DATASET_PATH = '/content/drive/MyDrive/DATASETS/dataset-5000/train_aggressive_cleaned.json'

print(f"📁 Dataset location: {DRIVE_DATASET_PATH}")
print("✅ Using DataTurks dataset (200 examples - HIGH QUALITY labels)")


Mounted at /content/drive
✅ Google Drive mounted!
📁 Dataset location: /content/drive/MyDrive/DATASETS/dataturks_resumes_fixed.json
✅ Using DataTurks dataset (email issues fixed, excellent quality)


## Step 3: Create Model Architecture (FROM SCRATCH)

In [3]:
%%writefile model_from_scratch.py
"""
BiLSTM-CRF model architecture for NER - trained completely from scratch.
No pre-trained models or weights used.
Includes advanced features: Character embeddings and Multi-head attention.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class CharCNN(nn.Module):
    """Character-level CNN for generating character embeddings."""
    def __init__(self, char_vocab_size=128, char_embed_dim=30, char_hidden_dim=50):
        super(CharCNN, self).__init__()
        self.char_embedding = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        
        # Multi-filter CNN (like Kim's CNN)
        self.conv1 = nn.Conv1d(char_embed_dim, char_hidden_dim, kernel_size=2, padding=1)
        self.conv2 = nn.Conv1d(char_embed_dim, char_hidden_dim, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(char_embed_dim, char_hidden_dim, kernel_size=4, padding=2)
        
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(char_hidden_dim * 3, char_hidden_dim)
        
    def forward(self, char_ids):
        # char_ids: [batch_size, seq_len, max_word_len]
        batch_size, seq_len, max_word_len = char_ids.shape
        char_ids = char_ids.view(-1, max_word_len)  # [batch*seq_len, max_word_len]
        
        char_emb = self.char_embedding(char_ids)  # [batch*seq_len, max_word_len, char_embed_dim]
        char_emb = char_emb.transpose(1, 2)  # [batch*seq_len, char_embed_dim, max_word_len]
        
        # Apply convolutions
        conv1_out = F.relu(self.conv1(char_emb))
        conv2_out = F.relu(self.conv2(char_emb))
        conv3_out = F.relu(self.conv3(char_emb))
        
        # Pooling
        pool1 = self.pool(conv1_out).squeeze(-1)
        pool2 = self.pool(conv2_out).squeeze(-1)
        pool3 = self.pool(conv3_out).squeeze(-1)
        
        # Concatenate and project
        char_out = torch.cat([pool1, pool2, pool3], dim=1)
        char_out = self.fc(char_out)
        
        # Reshape back
        char_out = char_out.view(batch_size, seq_len, -1)
        return char_out


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention mechanism."""
    def __init__(self, hidden_dim, num_heads=8, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert hidden_dim % num_heads == 0
        
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.q_linear = nn.Linear(hidden_dim, hidden_dim)
        self.k_linear = nn.Linear(hidden_dim, hidden_dim)
        self.v_linear = nn.Linear(hidden_dim, hidden_dim)
        self.out_linear = nn.Linear(hidden_dim, hidden_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, x, mask=None):
        # x: [batch_size, seq_len, hidden_dim]
        residual = x
        batch_size, seq_len, _ = x.shape
        
        # Linear projections
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        
        output = self.out_linear(attn_output)
        output = self.dropout(output)
        
        # Residual connection and layer norm
        output = self.layer_norm(output + residual)
        return output


class BiLSTM_CRF_NER(nn.Module):
    """
    Advanced Bidirectional LSTM + CRF for Named Entity Recognition.
    Supports character embeddings and multi-head attention.
    All weights initialized randomly - no pre-training.
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, 
                 num_layers=2, dropout=0.5, use_char_emb=False, use_attention=False):
        super(BiLSTM_CRF_NER, self).__init__()
        
        self.use_char_emb = use_char_emb
        self.use_attention = use_attention
        
        # Word embeddings (random initialization - FROM SCRATCH)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Character embeddings (optional)
        if use_char_emb:
            self.char_cnn = CharCNN(char_vocab_size=128, char_embed_dim=30, char_hidden_dim=50)
            input_dim = embedding_dim + 50  # Word + char embeddings
        else:
            input_dim = embedding_dim
        
        # Layer normalization for word embeddings
        self.embed_norm = nn.LayerNorm(embedding_dim)
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Multi-head attention (optional)
        if use_attention:
            self.attention = MultiHeadAttention(hidden_dim * 2, num_heads=8, dropout=dropout)
        
        # Feed-forward network with residual connection
        lstm_output_dim = hidden_dim * 2  # Bidirectional
        self.ffn = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_output_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_output_dim * 2, lstm_output_dim),
            nn.Dropout(dropout)
        )
        self.ffn_norm = nn.LayerNorm(lstm_output_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Linear layer to map to label space
        self.hidden2tag = nn.Linear(lstm_output_dim, num_labels)
        
    def forward(self, x, mask=None, char_ids=None):
        # x: [batch_size, seq_len]
        # char_ids: [batch_size, seq_len, max_word_len] (optional)
        
        # Word embeddings
        word_emb = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        word_emb = self.embed_norm(word_emb)  # Normalize word embeddings
        
        # Character embeddings (if enabled)
        if self.use_char_emb:
            if char_ids is not None:
                char_emb = self.char_cnn(char_ids)
                word_emb = torch.cat([word_emb, char_emb], dim=-1)
            else:
                # char_ids not provided, pad with zeros
                zeros = torch.zeros(word_emb.size(0), word_emb.size(1), 50, device=word_emb.device)
                word_emb = torch.cat([word_emb, zeros], dim=-1)
            
            # Normalize combined embeddings
            if not hasattr(self, 'combined_norm'):
                self.combined_norm = nn.LayerNorm(250).to(word_emb.device)
            word_emb = self.combined_norm(word_emb)
        
        word_emb = self.dropout(word_emb)
        
        # LSTM
        lstm_out, _ = self.lstm(word_emb)
        lstm_out = self.dropout(lstm_out)
        
        # Multi-head attention (if enabled)
        if self.use_attention:
            lstm_out = self.attention(lstm_out, mask=mask)
        
        # Feed-forward network with residual
        ffn_out = self.ffn(lstm_out)
        ffn_out = self.ffn_norm(ffn_out + lstm_out)
        ffn_out = self.dropout(ffn_out)
        
        # Final classification
        logits = self.hidden2tag(ffn_out)
        return logits


Writing model_from_scratch.py


## Step 4: Create Training Utilities

In [4]:
%%writefile train_utils_scratch.py
"""Training utilities for from-scratch model."""

import json
import torch
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm
import numpy as np


def load_new_dataset(json_file_path):
    """Load dataset in the new format."""
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    training_data = []
    for entry in data:
        if 'text' not in entry or 'annotations' not in entry:
            continue

        text = entry['text']
        annotations = entry['annotations']

        entities = []
        for ann in annotations:
            if not isinstance(ann, list) or len(ann) != 3:
                continue

            start, end, label = ann
            if start < 0 or end > len(text) or start >= end:
                continue

            entities.append((start, end, label))

        if entities:
            training_data.append((text, {"entities": entities}))

    return training_data


def build_vocab(data, min_freq=2):
    """Build vocabulary from training data."""
    word_freq = defaultdict(int)

    for text, _ in data:
        words = text.lower().split()
        for word in words:
            word_freq[word] += 1

    vocab = {'<PAD>': 0, '<UNK>': 1}
    idx = 2

    for word, freq in sorted(word_freq.items(), key=lambda x: x[1], reverse=True):
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1

    return vocab


def create_tag_mappings(data):
    """Create tag mappings with BIO format."""
    all_labels = set(['O'])

    for text, entities in data:
        for start, end, label in entities['entities']:
            all_labels.add(f'B-{label}')
            all_labels.add(f'I-{label}')

    tags = sorted(list(all_labels))
    tag2idx = {tag: idx for idx, tag in enumerate(tags)}
    idx2tag = {idx: tag for tag, idx in tag2idx.items()}

    return tag2idx, idx2tag


def text_to_indices(text, vocab, max_len=500):
    """Convert text to sequence of word indices."""
    words = text.lower().split()
    indices = [vocab.get(word, vocab['<UNK>']) for word in words]

    if len(indices) > max_len:
        indices = indices[:max_len]
    else:
        indices = indices + [vocab['<PAD>']] * (max_len - len(indices))

    return indices


def align_labels_with_words(text, entities, max_len=500):
    """Align entity labels with word positions."""
    words = text.lower().split()
    labels = ['O'] * len(words)

    # Map character positions to word positions
    char_to_word = {}
    char_pos = 0
    for word_idx, word in enumerate(words):
        for _ in range(len(word)):
            char_to_word[char_pos] = word_idx
            char_pos += 1
        char_pos += 1  # space

    # Assign labels
    for start, end, label in sorted(entities, key=lambda x: x[0]):
        if start in char_to_word and end-1 in char_to_word:
            start_word = char_to_word[start]
            end_word = char_to_word[end-1]
            for i in range(start_word, end_word + 1):
                if i < len(labels):
                    if i == start_word:
                        labels[i] = f'B-{label}'
                    else:
                        labels[i] = f'I-{label}'

    if len(labels) > max_len:
        labels = labels[:max_len]
    else:
        labels = labels + ['O'] * (max_len - len(labels))

    return labels


class ResumeDatasetScratch(Dataset):
    """Dataset for from-scratch training."""
    def __init__(self, data, vocab, tag2idx, max_len=500):
        self.data = data
        self.vocab = vocab
        self.tag2idx = tag2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, entities = self.data[idx]

        input_ids = text_to_indices(text, self.vocab, self.max_len)
        labels = align_labels_with_words(text, entities['entities'], self.max_len)
        label_ids = [self.tag2idx.get(label, self.tag2idx['O']) for label in labels]

        # Create attention mask (1 for real tokens, 0 for padding)
        words = text.lower().split()
        actual_len = min(len(words), self.max_len)
        attention_mask = [1] * actual_len + [0] * (self.max_len - actual_len)

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(label_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        }

Writing train_utils_scratch.py


## Step 5: Load Data and Build Vocabulary

In [5]:
import torch
from train_utils_scratch import load_new_dataset, build_vocab, create_tag_mappings, align_labels_with_words

# Load dataset
dataset_path = DRIVE_DATASET_PATH
print(f"Loading dataset from: {dataset_path}")
data = load_new_dataset(dataset_path)
print(f"✅ Loaded {len(data)} entries")

# Build vocabulary FROM YOUR DATA (not pre-trained)
print("\nBuilding vocabulary from training data...")
vocab = build_vocab(data, min_freq=2)
print(f"✅ Vocabulary size: {len(vocab)} (built from your data)")

# Create tag mappings
print("\nCreating tag mappings...")
tag2idx, idx2tag = create_tag_mappings(data)
print(f"✅ Number of labels: {len(tag2idx)}")
print(f"Sample labels: {list(tag2idx.keys())[:10]}")

# Split data
TRAIN_SPLIT = 0.9
split_idx = int(len(data) * TRAIN_SPLIT)
train_data = data[:split_idx]
val_data = data[split_idx:]
print(f"\n✅ Train: {len(train_data)} entries")
print(f"✅ Validation: {len(val_data)} entries")

Loading dataset from: /content/drive/MyDrive/DATASETS/dataturks_resumes_fixed.json
✅ Loaded 200 entries

Building vocabulary from training data...
✅ Vocabulary size: 6670 (built from your data)

Creating tag mappings...
✅ Number of labels: 23
Sample labels: ['B-', 'B-College Name', 'B-Companies worked at', 'B-Degree', 'B-Designation', 'B-Graduation Year', 'B-Location', 'B-Name', 'B-Skills', 'B-UNKNOWN']

✅ Train: 180 entries
✅ Validation: 20 entries


## Step 6: Initialize Model (FROM SCRATCH) - OPTIMIZED VERSION

**Key Optimizations Applied:**
1. **Larger Architecture**: 128 embedding dim (↑ from 100), 384 hidden dim (↑ from 256), 3 layers (↑ from 2)
2. **Learning Rate Warmup**: Gradually increases LR in first 3 epochs for stable training
3. **Weight Decay**: L2 regularization (1e-5) prevents overfitting
4. **Gradient Clipping**: Prevents exploding gradients during training
5. **Longer Training**: 50 epochs with patience=10 for better convergence
6. **Balanced Dropout**: 0.5 (was 0.6) - better balance between regularization and learning

**Expected Improvements:**
- Better accuracy (larger model capacity)
- More stable training (warmup + gradient clipping)
- Less overfitting (weight decay + balanced dropout)
- Better convergence (longer training with patience)

## Step 6: Initialize Model (FROM SCRATCH)

In [6]:
from model_from_scratch import BiLSTM_CRF_NER
from train_utils_scratch import ResumeDatasetScratch
from torch.utils.data import DataLoader
# Model hyperparameters (FURTHER OPTIMIZED)
EMBEDDING_DIM = 200  # Word embedding dimension (increased for better capacity)
HIDDEN_DIM = 512     # LSTM hidden dimension (increased for better capacity)
NUM_LAYERS = 4       # Number of LSTM layers (increased for deeper learning)
DROPOUT = 0.6        # Dropout rate (increased for better regularization)
MAX_LEN = 500        # Maximum sequence length
BATCH_SIZE = 8       # Batch size (smaller for more updates per epoch)
EPOCHS = 80          # Number of epochs (more training for small dataset)
LEARNING_RATE = 0.001  # Learning rate (slightly higher for faster initial learning)
USE_CLASS_WEIGHTING = True  # Enable class weighting for imbalanced data
USE_CHAR_EMB = True  # Enable character embeddings
USE_ATTENTION = True  # Enable multi-head attention
WEIGHT_DECAY = 1e-4  # L2 regularization (increased for stronger regularization)
GRADIENT_CLIP = 1.0  # Gradient clipping (prevents exploding gradients)
USE_WARMUP = True    # Learning rate warmup (helps early training)
WARMUP_EPOCHS = 3    # Number of warmup epochs
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*60)
print("SYSTEM STATUS")
print("="*60)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Using CPU - training will be slower")
print("="*60)
# Initialize model FROM SCRATCH (random weights)
print("\nInitializing model FROM SCRATCH...")
print("⚠️ No pre-trained models used - all weights are random!")
model = BiLSTM_CRF_NER(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_labels=len(tag2idx),
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
    use_char_emb=USE_CHAR_EMB,
    use_attention=USE_ATTENTION
)
model.to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model initialized with {num_params:,} parameters")
print(f"✅ All weights are RANDOM (no pre-training)")
# Create datasets
train_dataset = ResumeDatasetScratch(train_data, vocab, tag2idx, MAX_LEN)
val_dataset = ResumeDatasetScratch(val_data, vocab, tag2idx, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
print(f"\n✅ Training batches: {len(train_loader)}")
print(f"✅ Validation batches: {len(val_loader)}")

SYSTEM STATUS
Device: cuda
✅ GPU: Tesla T4

Initializing model FROM SCRATCH...
⚠️ No pre-trained models used - all weights are random!
✅ Model initialized with 9,540,631 parameters
✅ All weights are RANDOM (no pre-training)

✅ Training batches: 12
✅ Validation batches: 2


## Step 7: Train the Model

In [7]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from collections import Counter

# Calculate class weights for imbalanced data
if USE_CLASS_WEIGHTING:
    print("Calculating class weights for imbalanced data...")
    label_counts = Counter()

    for text, entities in train_data:
        labels = align_labels_with_words(text, entities['entities'], MAX_LEN)
        for label in labels:
            label_counts[label] += 1

    # Calculate weights (inverse frequency)
    total_labels = sum(label_counts.values())
    class_weights = torch.ones(len(tag2idx), device=device)

    for label, count in label_counts.items():
        if label in tag2idx and count > 0:
            # Weight = total / (num_classes * count)
            # This gives higher weight to rare classes
            weight = total_labels / (len(tag2idx) * count)
            class_weights[tag2idx[label]] = weight

    print(f"✅ Class weights calculated")
    print(f"   O label weight: {class_weights[tag2idx['O']]:.3f}")
    # Show top 5 rarest entities
    rarest = sorted(label_counts.items(), key=lambda x: x[1])[:5]
    for label, count in rarest:
        if label in tag2idx:
            print(f"   {label} weight: {class_weights[tag2idx[label]]:.3f} (count: {count})")
else:
    class_weights = None

# Loss and optimizer with class weighting
if USE_CLASS_WEIGHTING and class_weights is not None:
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print("✅ Using weighted CrossEntropyLoss (helps with class imbalance)")
else:
    criterion = nn.CrossEntropyLoss()
    print("✅ Using standard CrossEntropyLoss")

# Optimizer with weight decay (L2 regularization)
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY  # L2 regularization
)
print(f"✅ Optimizer: Adam with weight_decay={WEIGHT_DECAY}")

# Learning rate scheduler (reduce LR when validation loss plateaus)
# Note: verbose parameter not available in older PyTorch versions
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
print("✅ Learning rate scheduler enabled (reduces LR on plateau)")
print("   Will reduce LR by 50% if no improvement for 3 epochs")

def train_epoch(model, dataloader, optimizer, criterion, device, tag2idx):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids, mask=attention_mask)
        logits = logits.view(-1, logits.size(-1))
        labels_flat = labels.view(-1)
        mask_flat = attention_mask.view(-1)

        # Calculate loss (ignore padding tokens, not 'O' labels)
        # 'O' is a valid label, we only want to ignore actual padding
        active_loss = mask_flat == 1
        active_logits = logits[active_loss]
        active_labels = labels_flat[active_loss]

        if active_labels.numel() > 0:
            loss = criterion(active_logits, active_labels)
        else:
            loss = torch.tensor(0.0, device=device)

        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping (prevents exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP)
        optimizer.step()

        total_loss += loss.item()

        # Calculate accuracy (only on non-padding tokens, excluding 'O' for entity accuracy)
        predictions = torch.argmax(active_logits, dim=-1) if active_logits.numel() > 0 else torch.tensor([], device=device, dtype=torch.long)
        # For NER, we typically care about entity accuracy (not 'O' labels)
        entity_mask = (active_labels != tag2idx['O'])
        correct += ((predictions == active_labels) & entity_mask).sum().item()
        total += entity_mask.sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total if total > 0 else 0
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device, tag2idx):
    """Validate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits = model(input_ids, mask=attention_mask)
            logits = logits.view(-1, logits.size(-1))
            labels_flat = labels.view(-1)
            mask_flat = attention_mask.view(-1)

            # Calculate loss (ignore padding tokens)
            active_loss = mask_flat == 1
            active_logits = logits[active_loss]
            active_labels = labels_flat[active_loss]

            if active_labels.numel() > 0:
                loss = criterion(active_logits, active_labels)
            else:
                loss = torch.tensor(0.0, device=device)

            total_loss += loss.item()

            # Calculate accuracy (only on non-padding tokens, excluding 'O' for entity accuracy)
            predictions = torch.argmax(active_logits, dim=-1) if active_logits.numel() > 0 else torch.tensor([], device=device, dtype=torch.long)
            entity_mask = (active_labels != tag2idx['O'])
            correct += ((predictions == active_labels) & entity_mask).sum().item()
            total += entity_mask.sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total if total > 0 else 0
    return avg_loss, accuracy


# Training loop with early stopping
print(f"\n{'='*60}")
print("STARTING TRAINING FROM SCRATCH")
print(f"{'='*60}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"{'='*60}\n")

best_val_loss = float('inf')
USE_EARLY_STOPPING = True  # Enable early stopping to prevent overfitting
patience = 7  # Early stopping patience (reduced for faster stopping with small dataset)
patience_counter = 0

# Track training history
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 60)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, tag2idx)
    val_loss, val_acc = validate(model, val_loader, criterion, device, tag2idx)

    # Learning rate warmup (gradually increase LR in early epochs)
    if USE_WARMUP and epoch <= WARMUP_EPOCHS:
        warmup_lr = LEARNING_RATE * (epoch / WARMUP_EPOCHS)
        for param_group in optimizer.param_groups:
            param_group['lr'] = warmup_lr
        current_lr = warmup_lr
        print(f"🔥 Warmup epoch {epoch}/{WARMUP_EPOCHS} - LR: {current_lr:.6f}")
    else:
        # Update learning rate based on validation loss (after warmup)
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

    print(f"✅ Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")
    print(f"✅ Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")

    if not (USE_WARMUP and epoch <= WARMUP_EPOCHS):
        print(f"📊 Learning Rate: {current_lr:.6f}", end="")
        # Manually log LR reduction (since verbose isn't available)
        if current_lr < old_lr:
            print(f" ⬇️  (Reduced from {old_lr:.6f})")
        else:
            print()

    # Track history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'vocab': vocab,
            'tag2idx': tag2idx,
            'idx2tag': idx2tag,
            'embedding_dim': EMBEDDING_DIM,
            'hidden_dim': HIDDEN_DIM,
        }, '/content/model_from_scratch.bin')
        print(f"💾 Saved best model (val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⚠️ Early stopping triggered! No improvement for {patience} epochs.")
            print(f"Best validation loss: {best_val_loss:.4f}")
            print(f"Best validation accuracy: {max(val_accs):.4f}")
            break

print(f"\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")
print("✅ Model trained FROM SCRATCH (no pre-trained models used)")
print("✅ Model saved to: /content/model_from_scratch.bin")
print(f"{'='*60}")

Calculating class weights for imbalanced data...
✅ Class weights calculated
   O label weight: 0.047
   B-Years of Experience weight: 230.179 (count: 17)
   I-Years of Experience weight: 205.950 (count: 19)
   I-Graduation Year weight: 150.502 (count: 26)
   I-Location weight: 43.000 (count: 91)
   B-Graduation Year weight: 37.991 (count: 103)
✅ Using weighted CrossEntropyLoss (helps with class imbalance)
✅ Optimizer: Adam with weight_decay=1e-05
✅ Learning rate scheduler enabled (reduces LR on plateau)
   Will reduce LR by 50% if no improvement for 3 epochs

STARTING TRAINING FROM SCRATCH
Epochs: 50
Learning Rate: 0.001
Batch Size: 16


Epoch 1/50
------------------------------------------------------------


Validation: 100%|██████████| 2/2 [00:00<00:00, 13.64it/s]


🔥 Warmup epoch 1/3 - LR: 0.000333
✅ Train Loss: 2.9250, Train Accuracy: 0.0688
✅ Val Loss: 2.4095, Val Accuracy: 0.1488
💾 Saved best model (val_loss: 2.4095, val_acc: 0.1488)

Epoch 2/50
------------------------------------------------------------


Validation: 100%|██████████| 2/2 [00:00<00:00, 14.47it/s]


🔥 Warmup epoch 2/3 - LR: 0.000667
✅ Train Loss: 2.4893, Train Accuracy: 0.1681
✅ Val Loss: 2.0802, Val Accuracy: 0.3512
💾 Saved best model (val_loss: 2.0802, val_acc: 0.3512)

Epoch 3/50
------------------------------------------------------------


...

✅ Train Loss: 0.6542, Train Accuracy: 0.7596
✅ Val Loss: 1.3403, Val Accuracy: 0.4711
📊 Learning Rate: 0.000250 ⬇️  (Reduced from 0.000500)

Epoch 20/50
------------------------------------------------------------


Validation: 100%|██████████| 2/2 [00:00<00:00, 13.00it/s]


✅ Train Loss: 0.6231, Train Accuracy: 0.7604
✅ Val Loss: 1.2980, Val Accuracy: 0.4690
📊 Learning Rate: 0.000250

Epoch 21/50
------------------------------------------------------------


Validation: 100%|██████████| 2/2 [00:00<00:00, 13.51it/s]

✅ Train Loss: 0.6046, Train Accuracy: 0.7738
✅ Val Loss: 1.3049, Val Accuracy: 0.4835
📊 Learning Rate: 0.000250

⚠️ Early stopping triggered! No improvement for 10 epochs.
Best validation loss: 1.1123
Best validation accuracy: 0.6157

TRAINING COMPLETE!
✅ Model trained FROM SCRATCH (no pre-trained models used)
✅ Model saved to: /content/model_from_scratch.bin


## Step 8: Download Model

In [8]:
# Download model to your computer
from google.colab import files
files.download('/content/model_from_scratch.bin')
print("✅ Model downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model downloaded!


## Step 8: Test the Trained Model

Test your trained model on sample resume text to see extracted entities.

## Step 8: Test the Trained Model

Test your trained model on sample resume text to see extracted entities.

In [ ]:
# Test the trained model
import torch
from model_from_scratch import BiLSTM_CRF_NER
from train_utils_scratch import text_to_indices

print("=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

# Load the saved model
MODEL_PATH = '/content/model_from_scratch.bin'
print(f"\n📂 Loading model from: {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=device)
vocab = checkpoint['vocab']
tag2idx = checkpoint['tag2idx']
idx2tag = {v: k for k, v in tag2idx.items()}  # Reverse mapping

print(f"✅ Loaded vocabulary: {len(vocab)} words")
print(f"✅ Loaded {len(tag2idx)} entity labels")
print(f"Labels: {list(tag2idx.keys())[:10]}...")

# Get model config from checkpoint (with fallbacks)
embedding_dim = checkpoint.get('embedding_dim', EMBEDDING_DIM)
hidden_dim = checkpoint.get('hidden_dim', HIDDEN_DIM)
num_layers = checkpoint.get('num_layers', NUM_LAYERS)

print(f"✅ Model config: embedding_dim={embedding_dim}, hidden_dim={hidden_dim}, layers={num_layers}")

# Recreate model architecture
# Try loading with advanced features first
model = BiLSTM_CRF_NER(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_labels=len(tag2idx),
    num_layers=num_layers,
    dropout=0.0,  # No dropout during inference
    use_char_emb=USE_CHAR_EMB,  # Use same as training
    use_attention=USE_ATTENTION  # Use same as training
)

# Load state dict with strict=False to handle missing keys
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
print("✅ Model loaded successfully with advanced features")

model.to(device)
model.eval()

print("✅ Model ready for testing!\n")

# Sample resume text for testing
test_text = """
John Smith
Software Engineer at Google
Email: john.smith@email.com
Location: San Francisco, California

WORK EXPERIENCE
Senior Software Engineer
Google Inc.
5 years of experience in Python, Machine Learning, and Cloud Computing

EDUCATION
Bachelor of Science in Computer Science
Stanford University
2015-2019

SKILLS
Python, Java, TensorFlow, PyTorch, AWS, Docker, Kubernetes
"""

print("-" * 70)
print("TEST TEXT:")
print("-" * 70)
print(test_text)
print()

# Convert text to indices
input_ids = text_to_indices(test_text, vocab, MAX_LEN)
input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

# Get predictions
with torch.no_grad():
    # Create attention mask
    words = test_text.lower().split()
    actual_len = min(len(words), MAX_LEN)
    attention_mask = torch.tensor([[1] * actual_len + [0] * (MAX_LEN - actual_len)], dtype=torch.long).to(device)
    
    logits = model(input_tensor, mask=attention_mask)
    predictions = torch.argmax(logits, dim=-1)[0].cpu().numpy()  # [seq_len]
    probs = torch.softmax(logits[0], dim=-1)  # [seq_len, num_labels]

# Convert predictions to labels
words = test_text.lower().split()
predicted_labels = []
for i, pred_id in enumerate(predictions):
    if i < len(words):
        label = idx2tag[pred_id]
        confidence = probs[i][pred_id].item() * 100
        predicted_labels.append((words[i], label, confidence))

# Extract entities
print("=" * 70)
print("EXTRACTED ENTITIES:")
print("=" * 70)

entities = []
current_entity = None

for word, label, conf in predicted_labels:
    # Skip 'O' labels
    if label == 'O':
        if current_entity:
            entities.append(current_entity)
            current_entity = None
        continue
    
    # Handle BIO format
    if label.startswith('B-'):
        # Beginning of new entity
        if current_entity:
            entities.append(current_entity)
        entity_type = label[2:]  # Remove 'B-' prefix
        current_entity = {
            'entity': entity_type,
            'text': word,
            'words': [word],
            'confidence': conf
        }
    elif label.startswith('I-'):
        # Continuation of entity
        entity_type = label[2:]  # Remove 'I-' prefix
        if current_entity and current_entity['entity'] == entity_type:
            current_entity['text'] += ' ' + word
            current_entity['words'].append(word)
            current_entity['confidence'] = min(current_entity['confidence'], conf)  # Min confidence
        else:
            # New entity
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'entity': entity_type,
                'text': word,
                'words': [word],
                'confidence': conf
            }
    else:
        # Direct label (no BIO prefix)
        if current_entity:
            entities.append(current_entity)
        entities.append({
            'entity': label,
            'text': word,
            'words': [word],
            'confidence': conf
        })
        current_entity = None

# Add last entity if exists
if current_entity:
    entities.append(current_entity)

# Group entities by type
entities_by_type = {}
for ent in entities:
    entity_type = ent['entity']
    if entity_type not in entities_by_type:
        entities_by_type[entity_type] = []
    entities_by_type[entity_type].append({
        'text': ent['text'],
        'confidence': f"{ent['confidence']:.1f}%"
    })

# Display results
if entities_by_type:
    for entity_type in sorted(entities_by_type.keys()):
        print(f"\n📌 {entity_type}:")
        # Remove duplicates while preserving order
        seen = set()
        for item in entities_by_type[entity_type]:
            if item['text'] not in seen:
                seen.add(item['text'])
                print(f"   • {item['text']} (confidence: {item['confidence']})")
else:
    print("⚠️ No entities extracted.")
    print("   This could mean:")
    print("   - The model needs more training")
    print("   - The text format doesn't match training data")
    print("   - Try testing with text similar to your training data")

print(f"\n✅ Total entities extracted: {len(entities)}")
print(f"✅ Entity types found: {len(entities_by_type)}")

# Show word-level predictions (first 30 words)
print("\n" + "-" * 70)
print("WORD-LEVEL PREDICTIONS (First 30 words):")
print("-" * 70)
for i, (word, label, conf) in enumerate(predicted_labels[:30]):
    if label != 'O':
        print(f"  {word:20s} → {label:25s} ({conf:.1f}%)")
    elif i < 10:  # Show some 'O' labels for context
        print(f"  {word:20s} → {label}")

print("\n" + "=" * 70)
print("✅ Testing complete!")
print("=" * 70)
print("\n💡 Tip: Modify the 'test_text' variable above to test on your own resume text.")

In [ ]:
# Test the trained model
import torch
from model_from_scratch import BiLSTM_CRF_NER
from train_utils_scratch import text_to_indices, align_labels_with_words
import json

print("=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

# Load the saved model
MODEL_PATH = '/content/model_from_scratch.bin'
print(f"\n📂 Loading model from: {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=device)
vocab = checkpoint['vocab']
tag2idx = checkpoint['tag2idx']
idx2tag = checkpoint['idx2tag']
embedding_dim = checkpoint['embedding_dim']
hidden_dim = checkpoint['hidden_dim']

print(f"✅ Loaded vocabulary: {len(vocab)} words")
print(f"✅ Loaded {len(tag2idx)} entity labels")
print(f"✅ Model config: embedding_dim={embedding_dim}, hidden_dim={hidden_dim}")

# Recreate model architecture
model = BiLSTM_CRF_NER(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_labels=len(tag2idx),
    num_layers=NUM_LAYERS,
    dropout=0.0,  # No dropout during inference
    use_char_emb=USE_CHAR_EMB,  # Use same as training
    use_attention=USE_ATTENTION  # Use same as training
)
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
model.to(device)
model.eval()

print("✅ Model loaded successfully with advanced features")

print("✅ Model loaded and ready for testing!\n")

# Sample resume text for testing
test_text = """
John Smith
Software Engineer at Google
Email: john.smith@email.com
Location: San Francisco, California

WORK EXPERIENCE
Senior Software Engineer
Google Inc.
5 years of experience in Python, Machine Learning, and Cloud Computing

EDUCATION
Bachelor of Science in Computer Science
Stanford University
2015-2019

SKILLS
Python, Java, TensorFlow, PyTorch, AWS, Docker, Kubernetes
"""

print("-" * 70)
print("TEST TEXT:")
print("-" * 70)
print(test_text)
print()

# Convert text to indices
input_ids = text_to_indices(test_text, vocab, MAX_LEN)
input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

# Get predictions
with torch.no_grad():
    # Create attention mask
    words = test_text.lower().split()
    actual_len = min(len(words), MAX_LEN)
    attention_mask = torch.tensor([[1] * actual_len + [0] * (MAX_LEN - actual_len)], dtype=torch.long).to(device)
    
    logits = model(input_tensor, mask=attention_mask)
    predictions = torch.argmax(logits, dim=-1)[0].cpu().numpy()  # [seq_len]

# Convert predictions to labels
words = test_text.lower().split()
predicted_labels = []
for i, pred_id in enumerate(predictions):
    if i < len(words):
        label = idx2tag[pred_id]
        predicted_labels.append((words[i], label))

# Extract entities
print("=" * 70)
print("EXTRACTED ENTITIES:")
print("=" * 70)

entities = []
current_entity = None

for word, label in predicted_labels:
    # Skip 'O' labels
    if label == 'O':
        if current_entity:
            entities.append(current_entity)
            current_entity = None
        continue
    
    # Handle BIO format
    if label.startswith('B-'):
        # Beginning of new entity
        if current_entity:
            entities.append(current_entity)
        entity_type = label[2:]  # Remove 'B-' prefix
        current_entity = {
            'entity': entity_type,
            'text': word,
            'words': [word]
        }
    elif label.startswith('I-'):
        # Continuation of entity
        entity_type = label[2:]  # Remove 'I-' prefix
        if current_entity and current_entity['entity'] == entity_type:
            current_entity['text'] += ' ' + word
            current_entity['words'].append(word)
        else:
            # New entity (shouldn't happen with proper BIO, but handle it)
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'entity': entity_type,
                'text': word,
                'words': [word]
            }
    else:
        # Direct label (no BIO prefix)
        if current_entity:
            entities.append(current_entity)
        entities.append({
            'entity': label,
            'text': word,
            'words': [word]
        })
        current_entity = None

# Add last entity if exists
if current_entity:
    entities.append(current_entity)

# Group entities by type
entities_by_type = {}
for ent in entities:
    entity_type = ent['entity']
    if entity_type not in entities_by_type:
        entities_by_type[entity_type] = []
    entities_by_type[entity_type].append(ent['text'])

# Display results
if entities_by_type:
    for entity_type in sorted(entities_by_type.keys()):
        unique_entities = list(set(entities_by_type[entity_type]))  # Remove duplicates
        print(f"\n{entity_type}:")
        for text in unique_entities:
            print(f"  • {text}")
else:
    print("⚠️ No entities extracted. The model may need more training or the text format doesn't match training data.")

print(f"\n✅ Total entities extracted: {len(entities)}")
print(f"✅ Entity types found: {len(entities_by_type)}")

# Show prediction confidence (optional)
print("\n" + "-" * 70)
print("PREDICTION CONFIDENCE (Top 5 most confident predictions):")
print("-" * 70)
with torch.no_grad():
    probs = torch.softmax(logits[0], dim=-1)  # [seq_len, num_labels]
    top_probs, top_indices = torch.topk(probs, k=1, dim=-1)
    
    for i, (word, label) in enumerate(predicted_labels[:20]):  # Show first 20
        if label != 'O':
            conf = top_probs[i][0].item() * 100
            print(f"  {word:20s} → {label:25s} ({conf:.1f}% confidence)")

print("\n" + "=" * 70)
print("✅ Testing complete!")
print("=" * 70)
print("\n💡 Tip: Modify the 'test_text' variable above to test on your own resume text.")